# Aufgabe 6 (Analyse von Stromtarif-Angeboten für Endkunden)

**In dieser Aufgabe sollen Neukunden-Angebotspreise für Endkunden in verschiedenen bayerischen Städten aus dem Jahr 2024 analysiert werden, wie sie beispielsweise auf Preisvergleichsportalen zu finden sind. Es ist pro Tag und Stadt eine JSON-Datei gegeben, in der bis zu 20 Tarifangebote aufgeführt sind, die an diesem Tag in dieser Stadt am günstigsten waren (Sortierung nach `Preis im 1. Jahr`) Die angebotenen Tarife beziehen sich jeweils auf einen jährlichen Gesamtverbrauch von 4000 kWh/Jahr. \
 a) Führen Sie die gegebenen Preisvergleichdaten in einem DataFrame namens `df_cust` zusammen. Exportieren Sie diesen als CSV-Datei namens `prices_customers.csv` und
 laden Sie diese mit Ihrer Abgabe auf Moodle hoch. Verwerfen Sie bitte zur Minimierung der Dateigröße alle Spalten, die im weiteren Verlauf nicht mehr verwendet werden. Kommentieren Sie nun den Code zur Datensatzgenerierung aus und lesen Sie die CSV-Datei in den DataFrame `df_cust` erneut aus dieser Datei ein. \
 b) Bereiten Sie die Daten auf die weitere Analyse vor, indem Sie geeignete Datentransformations- und-bereinigungsschritte durchführen.**

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openmeteo_requests
import requests_cache
from retry_requests import retry
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import HuberRegressor
import glob

In [2]:
'''def join_json_files_to_dataframe(base_folder):
    # Initialize an empty list to store data from all JSON files
    data_list = []
    
    # Recursively find all JSON files in the base folder
    for file_path in glob.glob(os.path.join(base_folder, '**', '*.json'), recursive=True):
        try:
            # Read each JSON file into a DataFrame and append to the list
            data = pd.read_json(file_path, lines=True)  # Use lines=True for JSONL files
            data_list.append(data)
        except ValueError as e:
            print(f"Error reading {file_path}: {e}")
    
    # Concatenate all DataFrames into a single DataFrame
    if data_list:
        combined_df = pd.concat(data_list, ignore_index=True)
        return combined_df
    else:
        print("No JSON files found or all files were invalid.")
        return pd.DataFrame()  # Return an empty DataFrame if no data

# Example usage
base_folder = "DAT-WS2425-Projektarbeit/DAT-WS2425-Projektarbeit/Daten/Endkundenpreise"
df_cust = join_json_files_to_dataframe(base_folder)

# Display the resulting DataFrame
df_cust'''

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,"{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '..."
1,"{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '..."
2,"{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '..."
3,"{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrau

In [3]:
#df_cust.to_csv("df_cust.csv", encoding="utf-8", index=False) 

In [4]:
'''df_cust = pd.read_csv('df_cust.csv')
df_cust'''

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,"{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '...","{'Postleitzahl': '80331', 'Jahresverbrauch': '..."
1,"{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '...","{'Postleitzahl': '83024', 'Jahresverbrauch': '..."
2,"{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '...","{'Postleitzahl': '84028', 'Jahresverbrauch': '..."
3,"{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrauch': '...","{'Postleitzahl': '86150', 'Jahresverbrau

In [5]:
'''import ast
for col in df_cust.columns:
    df_cust[col] = df_cust[col].apply(ast.literal_eval)

combined_df = pd.DataFrame()

for column in df_cust.columns:
    temp_df = pd.json_normalize(df_cust[column])
    combined_df = pd.concat([combined_df, temp_df], ignore_index=True)
    
combined_df'''

,Postleitzahl,Jahresverbrauch,Abschlagszahlung,Grundpreis,Arbeitspreis,Preisgarantie,Vertragslaufzeit,Verlängerung,Kündigungsfrist,Preis im 1. Jahr*,...,Neukundenbonus,Sofortbonus,Preis im 1. Jahr,Grundpreisrabatt:,Arbeitspreisrabatt,Grundpreisrabatt,Blitzbonus,Abschlagsrabatt,Zusätzlicher Aktionsbonus,Winterprämie
0,80331,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,61 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"79,77 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,83024,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,74 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"80,20 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,84028,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","20,32 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"82,14 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,86150,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,19 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"78,37 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,89233,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","22,02 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"87,80 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256335,95478,4.000 kWh,monatlich,"14,78 €/Monat (177,31 €/Jahr)","31,43 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,214 €,55 €,"97,13 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256336,95643,4.000 kWh,monatlich,"15,57 €/Monat (186,83 €/Jahr)","33,74 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,1 Monat,1 Monat,NaN,...,230 €,NaN,"108,84 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256337,96050,4.000 kWh,monatlich,"14,31 €/Monat (171,76 €/Jahr)","31,73 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,216 €,58 €,"97,25 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256338,96450,4.000 kWh,monatlich,"15,35 €/Monat (184,22 €/Jahr)","31,84 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,218 €,55 €,"98,74 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
#combined_df.to_csv("final_df.csv", encoding="utf-8", index=False)

In [7]:
final_df = pd.read_csv('final_df.csv', low_memory=False)
final_df

,Postleitzahl,Jahresverbrauch,Abschlagszahlung,Grundpreis,Arbeitspreis,Preisgarantie,Vertragslaufzeit,Verlängerung,Kündigungsfrist,Preis im 1. Jahr*,...,Neukundenbonus,Sofortbonus,Preis im 1. Jahr,Grundpreisrabatt:,Arbeitspreisrabatt,Grundpreisrabatt,Blitzbonus,Abschlagsrabatt,Zusätzlicher Aktionsbonus,Winterprämie
0,80331.0,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,61 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"79,77 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,83024.0,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,74 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"80,20 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,84028.0,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","20,32 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"82,14 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,86150.0,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","19,19 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"78,37 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,89233.0,4.000 kWh,monatlich,"14,40 €/Monat (172,86 €/Jahr)","22,02 Cent/kWh",1 Monat Preisfixierung,1 Monat,1 Monat,1 Monat,"87,80 €/Monat",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256335,95478.0,4.000 kWh,monatlich,"14,78 €/Monat (177,31 €/Jahr)","31,43 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,214 €,55 €,"97,13 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256336,95643.0,4.000 kWh,monatlich,"15,57 €/Monat (186,83 €/Jahr)","33,74 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,1 Monat,1 Monat,NaN,...,230 €,NaN,"108,84 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256337,96050.0,4.000 kWh,monatlich,"14,31 €/Monat (171,76 €/Jahr)","31,73 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,216 €,58 €,"97,25 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN
256338,96450.0,4.000 kWh,monatlich,"15,35 €/Monat (184,22 €/Jahr)","31,84 Cent/kWh",12 Monate Nettopreisgarantie,12 Monate,4 Wochen,4 Wochen,NaN,...,218 €,55 €,"98,74 €/Monat",NaN,NaN,NaN,NaN,NaN,NaN,NaN


**c) Wie viele verschiedene Tarife wurden insgesamt angeboten?**

In [8]:
print(len(final_df['Tarif'].unique()))

112


**Zu wie vielen Tagen sind pro Stadt Daten vorhanden?**

In [9]:
final_df.groupby('Stadt').count()/20

,Postleitzahl,Jahresverbrauch,Abschlagszahlung,Grundpreis,Arbeitspreis,Preisgarantie,Vertragslaufzeit,Verlängerung,Kündigungsfrist,Preis im 1. Jahr*,...,Neukundenbonus,Sofortbonus,Preis im 1. Jahr,Grundpreisrabatt:,Arbeitspreisrabatt,Grundpreisrabatt,Blitzbonus,Abschlagsrabatt,Zusätzlicher Aktionsbonus,Winterprämie
Stadt,,,,,,,,,,,,,,,,,,,,,
Amberg,321.95,321.95,321.95,321.95,321.95,321.95,321.95,321.95,321.95,78.45,...,234.60,206.65,243.50,2.45,26.05,9.50,11.45,2.80,1.50,0.90
Ansbach,319.05,319.05,319.05,319.05,319.05,319.05,319.05,319.05,319.05,81.95,...,228.40,192.95,237.10,2.25,26.90,8.70,12.80,2.15,0.90,1.10
Aschaffenburg,319.65,319.65,319.65,319.65,319.65,319.65,319.65,319.65,319.65,72.70,...,238.60,203.55,246.95,0.90,26.40,4.50,12.65,2.00,0.75,1.45
Augsburg,320.55,320.55,320.55,320.55,320.55,320.55,320.55,320.55,320.55,79.75,...,233.45,208.40,240.80,1.65,24.90,9.50,11.15,1.85,0.70,1.30
Bamberg,319.15,319.15,319.15,319.15,319.15,319.15,319.15,319.15,319.15,82.05,...,228.30,198.15,237.10,2.25,22.90,4.60,12.25,2.05,0.30,1.30
Bayreuth,318.75,318.75,318.75,318.75,318.75,318.75,318.75,318.75,318.75,81.30,...,229.25,201.85,237.45,2.90,30.00,10.55,11.55,1.85,0.60,0.85
Beilngries,320.60,320.60,320.60,320.60,320.60,320.60,320.60,320.60,320.60,82.50,...,230.75,199.85,238.10,2.50,19.55,5.95,10.55,2.95,0.70,1.85
Burglengenfeld,320.35,320.35,320.35,320.35,320.35,320.35,320.35,320.35,320.35,80.55,...,234.10,204.10,239.80,3.10,20.05,6.20,9.45,2.75,0.65,1.75
Cham,317.70,317.70,317.70,317.70,317.70,317.70,317.70,317.70,317.70,80.90,...,222.45,192.10,236.80,3.10,30.05,0.85,14.40,6.65,0.65,2.40


**Wie viele verschiedene Anbieter haben insgesamt Tarife angeboten?**

In [10]:
print(len(final_df['Anbieter'].unique()))

78


**d)  Ermitteln Sie, welche unterschiedlichen Tarife in Amberg angeboten wurden**

In [11]:
amberg_tarif=pd.Series(final_df[final_df['Stadt']=='Amberg']['Tarif'].unique(), name='Tarif')
amberg_tarif

0                                   Tarif Fairpower one
1                               Tarif Dynamischer Tarif
2                             Tarif ÖkoStrom Komfort 12
3     Marke der Rheinische Elektrizitäts- und Gasver...
4                       Tarif ELEKTRIZITÄT BERLIN Strom
                            ...                        
72                                 Tarif Q.ENERGY Eco12
73                              Tarif MONTANA garant 12
74                             Tarif Select+ Comfort 18
75    Marke der rhenag Rheinische Energie AG\n      ...
76                           Tarif Naturstrom Premium18
Name: Tarif, Length: 77, dtype: object